# Stage 0a - Convert NFP Scan to HDF5

## Parameters

In [ ]:
# --- paths ----------------------------------------------------------
# From config_step0.conf, which holds every path for the NFP scan.
# Edit that file, not this cell.
from holotomocupy.config import parse_args_coded_step0
_cfg = parse_args_coded_step0('config_step0.conf')
WITH_CA = _cfg.path   # processed data for this sample
RAW_NFP_CA = _cfg.raw_nfp   # raw NFP scan (read-only)


In [2]:
import numpy as np
import h5py
import os
import matplotlib.pyplot as plt
%matplotlib inline

# Scan folder
path = RAW_NFP_CA

# Output
path_out = WITH_CA + '/'
os.makedirs(path_out, exist_ok=True)
fname_out = f'{WITH_CA}/data_nfp.h5'

# Scan dimensions
n    = 2048      # detector size (pixels)
npos = 18 * 18   # total number of scan positions

# Optical / geometry parameters
sx0                     = 1.286e-3          # [m] position of the focus
z1c                     = -17.75e-3         # [m] position of the coded aperture
detector_pixelsize      = 3.03751e-6        # [m] detector pixel size
energy                  = 33.35             # [keV]
wavelength              = 1.24e-9 / energy  # [m]
focusToDetectorDistance = 1.28              # [m] focus–to-detector distance

# Cone beam adjustment
z1c           = z1c - sx0                                   # remove the offset so that the focus is at offset                                   
z1            = z1c                                         # [m] sample-to-focus distance. the sample is the coded aperture itself
z2            = focusToDetectorDistance - z1                # [m] sample-to-detector distance
distances     = (z1 * z2) / focusToDetectorDistance         # [m] effective Fresnel propagation distance
magnification = focusToDetectorDistance / z1                # can be negative depending on the sign of z1
voxelsize     = np.abs(detector_pixelsize / magnification)  # [m] object voxel size

print(f'z1            = {z1*1e3:.4f} mm')
print(f'z2            = {z2*1e3:.4f} mm')
print(f'distances     = {distances*1e3:.4f} mm  (effective Fresnel propagation)')
print(f'magnification = {magnification:.2f}  (|M| = {abs(magnification):.2f})')
print(f'voxelsize     = {voxelsize*1e9:.3f} nm')

PermissionError: [Errno 13] Permission denied: '/data2'

## Read data, flat-field, and dark frames

In [ ]:
# NFP data frames
with h5py.File(f'{path}/code2um_nfp18x18_010000.h5') as fid:
    data = fid['/entry_0000/measurement/data'][:].astype('float32') 
    # data.shape = (npos, n, n) = (324, 2048, 2048)

# Flat-field (reference, i.e. beam on, no sample/CA) frames 
with h5py.File(f'{path}/ref_0000.h5') as fid:
    ref  = fid['/entry_0000/measurement/data'][:].astype('float32')
    # ref.shape  = (21, 2048, 2048)

# Dark frames (shutter closed)
with h5py.File(f'{path}/dark_0000.h5') as fid:
    dark = fid['/entry_0000/measurement/data'][:].astype('float32')
    # dark.shape = (20, 2048, 2048)

print(f'data  shape : {data.shape}   dtype: {data.dtype}')
print(f'ref   shape : {ref.shape}    dtype: {ref.dtype}')
print(f'dark  shape : {dark.shape}   dtype: {dark.dtype}')
print(f'data  range : [{data.min():.0f}, {data.max():.0f}]')
print(f'ref   range : [{ref.min():.0f},  {ref.max():.0f}]')
print(f'dark  range : [{dark.min():.0f},   {dark.max():.0f}]')

data  shape : (324, 2048, 2048)   dtype: float32
ref   shape : (21, 2048, 2048)    dtype: float32
dark  shape : (20, 2048, 2048)   dtype: float32
data  range : [1400, 65535]
ref   range : [1676,  65535]
dark  range : [845,   65535]


## Save to exchange HDF5

In [ ]:
with h5py.File(fname_out, 'w') as fid:
    # Image data 
    fid.create_dataset('/exchange/data',       data=data)   # (324, 2048, 2048) float32
    fid.create_dataset('/exchange/data_white', data=ref)    # (21,  2048, 2048) float32
    fid.create_dataset('/exchange/data_dark',  data=dark)   # (20,  2048, 2048) float32

    # Geometry scalars
    fid.create_dataset('/exchange/voxelsize',             data=np.array([voxelsize],    dtype='float32'))
    fid.create_dataset('/exchange/z1',                    data=np.array([z1],           dtype='float32'))
    fid.create_dataset('/exchange/detector_pixelsize',    data=np.array([detector_pixelsize], dtype='float32'))
    fid.create_dataset('/exchange/focusdetectordistance', data=np.array([focusToDetectorDistance], dtype='float32'))
    fid.create_dataset('/exchange/energy',                data=np.array([energy],       dtype='float32'))

print(f'Saved -> {fname_out}')
print(f'  /exchange/data       : {data.shape}  dtype={data.dtype}')
print(f'  /exchange/data_white : {ref.shape}   dtype={ref.dtype}')
print(f'  /exchange/data_dark  : {dark.shape}  dtype={dark.dtype}')

Saved -> /data2/cekmekci/ESRF/AtomiumS2/with_ca/data_nfp.h5
  /exchange/data       : (324, 2048, 2048)  dtype=float32
  /exchange/data_white : (21, 2048, 2048)   dtype=float32
  /exchange/data_dark  : (20, 2048, 2048)  dtype=float32
